In [ ]:
import ee

ee.Authenticate()
ee.Initialize(project='avilight-483312')

# --- AOI: whole Philippines ---
philippines_fc = (
    ee.FeatureCollection('FAO/GAUL/2015/level0')
    .filter(ee.Filter.eq('ADM0_NAME', 'Philippines'))
)
AOI = philippines_fc.geometry()

# --- Grid resolution ---
# 0.05 degrees is about 5 km. It is practical for a country-wide export.
# 0.005 is much finer, but can create millions of cells and a very large CSV.
GRID_RES = 0.05

# --- Time range ---
# MCD12Q1 currently has data through 2024.
START_DATE = '2014-01-01'
END_DATE = '2025-01-01'  # includes 2014 through 2024

# --- Export settings ---
DRIVE_FOLDER = 'Satellite Datasets'
EXPORT_NAME = 'Philippines_LC_Type1_2014_2024'


def create_grid(aoi, res):
    """Create grid cells that intersect the Philippines."""
    bounds = ee.List(aoi.bounds().coordinates().get(0))

    lon_min = ee.Number(ee.List(bounds.get(0)).get(0))
    lat_min = ee.Number(ee.List(bounds.get(0)).get(1))
    lon_max = ee.Number(ee.List(bounds.get(2)).get(0))
    lat_max = ee.Number(ee.List(bounds.get(2)).get(1))

    lons = ee.List.sequence(lon_min, lon_max.subtract(res), res)
    lats = ee.List.sequence(lat_min, lat_max.subtract(res), res)

    def make_cell(lon, lat):
        lon = ee.Number(lon)
        lat = ee.Number(lat)

        geom = ee.Geometry.Rectangle(
            [lon, lat, lon.add(res), lat.add(res)],
            proj='EPSG:4326',
            geodesic=False
        )

        return ee.Feature(geom).set({
            'cell_id': (
                ee.String('cell_')
                .cat(lon.format('%.4f'))
                .cat('_')
                .cat(lat.format('%.4f'))
            ),
            'longitude': lon.add(res / 2),
            'latitude': lat.add(res / 2)
        })

    grid = (
        lons
        .map(lambda lon: lats.map(
            lambda lat: make_cell(lon, lat)
        ))
        .flatten()
    )

    return ee.FeatureCollection(grid).filterBounds(aoi)


grid_fc = create_grid(AOI, GRID_RES)

# --- MODIS yearly IGBP land-cover classes at 500 m ---
lc_ic = (
    ee.ImageCollection('MODIS/061/MCD12Q1')
    .filterDate(START_DATE, END_DATE)
    .select('LC_Type1')
)


def reduce_lc(image):
    """Calculate the modal land-cover class in every grid cell for one year."""
    year = ee.Date(image.get('system:time_start')).get('year')

    stats = image.reduceRegions(
        collection=grid_fc,
        reducer=ee.Reducer.mode(),
        scale=500
    )

    def format_row(feature):
        return feature.set({
            'year': year,
            'land_cover': feature.get('mode')
        })

    return (
        stats
        .map(format_row)
        .select([
            'cell_id',
            'year',
            'longitude',
            'latitude',
            'land_cover'
        ])
    )


# Create one record per grid cell per available year.
panel_fc = lc_ic.map(reduce_lc).flatten()

# Exclude grid cells that did not receive a land-cover class.
final_export_fc = panel_fc.filter(
    ee.Filter.notNull(['land_cover'])
)

# --- Export CSV ---
task = ee.batch.Export.table.toDrive(
    collection=final_export_fc,
    description=EXPORT_NAME,
    folder=DRIVE_FOLDER,
    fileNamePrefix=EXPORT_NAME,
    fileFormat='CSV'
)

task.start()

print('✅ Export started')
print('Task ID:', task.id)
print('📁 Google Drive folder:', DRIVE_FOLDER)
print('🗺️ AOI: Philippines')
print('📅 Available land-cover years: 2014–2024')
print('📏 Grid resolution:', GRID_RES, 'degrees')